In [3]:
# ==============================================================================
# ステップ 3：データ戦略評価（n_ vs m_ パフォーマンス比較）
# ==============================================================================
library(ggplot2)
library(tidyr)
library(dplyr)

# 1. パスの設定
dir_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/OOD_check/gcvEarth/"
summary_file <- "fixed20250702_gcvEarth_summary_20251212.csv"
target_path <- paste0(dir_path, summary_file)
today <- format(Sys.Date(), "%Y%m%d")

if (!file.exists(target_path)) {
    stop("サマリーCSVが見つかりません。パスを確認してください。")
}

# データのロード（check.names = FALSE で列名の .csv を維持）
df_summary <- read.csv(target_path, row.names = 1, check.names = FALSE)

# 2. 比較対象の抽出 (PCEmaxに関する内挿 R2 と 外挿 OOD_R2)
target_metrics <- c("R2_PCEmax", "OOD_R2_PCEmax")
target_datasets <- c("n_base_FP.csv", "m_base_FP.csv", "n_all_FP.csv", "m_all_FP.csv")

# データの整形
df_comp <- as.data.frame(t(df_summary[target_metrics, target_datasets]))
df_comp$dataset_name <- rownames(df_comp)

# 3. 解析結果のCSV出力
csv_step3_file <- paste0("step3_strategic_comparison_fixed_", today, ".csv")
write.csv(df_comp, csv_step3_file, row.names = FALSE)

# 4. グラフ用データの整形（ワイドからロング形式へ）
df_plot <- df_comp %>%
  pivot_longer(cols = all_of(target_metrics), 
               names_to = "metric_type", 
               values_to = "score") %>%
  mutate(
    strategy = ifelse(grepl("^n_", dataset_name), "measured_n", "imputed_m"),
    domain   = ifelse(grepl("_all_", dataset_name), "all_domain", "base_domain"),
    metric   = ifelse(metric_type == "R2_PCEmax", "interpolation_cv", "extrapolation_ood")
  )

# 5. 比較グラフの作成（スネークケース見出し）
p3 <- ggplot(df_plot, aes(x = domain, y = score, fill = strategy)) +
  geom_bar(stat = "identity", position = "dodge", alpha = 0.85) +
  facet_wrap(~ metric) +
  scale_fill_manual(values = c("measured_n" = "#2C3E50", "imputed_m" = "#E74C3C")) +
  theme_minimal(base_size = 14) +
  labs(
    title = "impact_of_data_imputation_on_fixed_model_performance",
    subtitle = "comparison_between_measured_n_and_imputed_m_datasets",
    x = "dataset_domain_scope",
    y = "r2_score_pce_max",
    fill = "data_strategy"
  ) +
  theme(strip.background = element_rect(fill = "gray92"),
        legend.position = "bottom")

# 6. PDF出力
pdf_step3_file <- paste0("step3_imputation_strategy_comparison_plot_", today, ".pdf")
ggsave(pdf_step3_file, p3, width = 10, height = 6)

# --- コンソールへの最終レポート ---
cat("====================================================\n")
cat("  ステップ 3：データ戦略解析完了レポート\n")
cat("====================================================\n")
print(df_comp)

cat("\n--- 解析の要点 (n_all_FP vs m_all_FP) ---\n")
n_ood <- df_comp["n_all_FP.csv", "OOD_R2_PCEmax"]
m_ood <- df_comp["m_all_FP.csv", "OOD_R2_PCEmax"]
cat("実測(n)の外挿精度: ", n_ood, "\n")
cat("補完(m)の外挿精度: ", m_ood, "\n")

if(n_ood > m_ood) {
  cat("\n考察: 実測データ(n)の優位性が証明されました。\n")
  cat("博士論文の結論: 'データの量（補完）より質（実測）が外挿性能を支配する'\n")
}

cat("\n--- 出力ファイル ---\n")
cat("1. CSV: ", csv_step3_file, "\n")
cat("2. PDF: ", pdf_step3_file, "\n")
cat("====================================================\n")

  <U+30B9><U+30C6><U+30C3><U+30D7> 3<U+FF1A><U+30C7><U+30FC><U+30BF><U+6226><U+7565><U+89E3><U+6790><U+5B8C><U+4E86><U+30EC><U+30DD><U+30FC><U+30C8>
              R2_PCEmax OOD_R2_PCEmax  dataset_name
n_base_FP.csv     0.800         0.010 n_base_FP.csv
m_base_FP.csv     0.679         0.091 m_base_FP.csv
n_all_FP.csv      0.801         0.745  n_all_FP.csv
m_all_FP.csv      0.679         0.091  m_all_FP.csv

--- <U+89E3><U+6790><U+306E><U+8981><U+70B9> (n_all_FP vs m_all_FP) ---
<U+5B9F><U+6E2C>(n)<U+306E><U+5916><U+633F><U+7CBE><U+5EA6>:  0.745 
<U+88DC><U+5B8C>(m)<U+306E><U+5916><U+633F><U+7CBE><U+5EA6>:  0.091 

<U+8003><U+5BDF>: <U+5B9F><U+6E2C><U+30C7><U+30FC><U+30BF>(n)<U+306E><U+512A><U+4F4D><U+6027><U+304C><U+8A3C><U+660E><U+3055><U+308C><U+307E><U+3057><U+305F><U+3002>
<U+535A><U+58EB><U+8AD6><U+6587><U+306E><U+7D50><U+8AD6>: '<U+30C7><U+30FC><U+30BF><U+306E><U+91CF><U+FF08><U+88DC><U+5B8C><U+FF09><U+3088><U+308A><U+8CEA><U+FF08><U+5B9F><U+6E2C><U+FF09><U+304C><U+5916><U+633F><U